## Content Based Recommender System

In [1]:
#1: import data 
#2: Data Preprocessing
#3: Train model
#4: Website
#4: Deploy

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [3]:
anime = pd.read_csv(
    "popular_anime.csv")

In [4]:
anime.head(2)

,id,name,genres,type,episodes,status,aired_from,aired_to,duration_per_ep,score,scored_by,rank,rating,studios,producers,image,trailer,synopsis
0,52991,Frieren: Beyond Journey's End,"Adventure, Drama, Fantasy",TV,28.0,Finished Airing,2023-09-29T00:00:00+00:00,2024-03-22T00:00:00+00:00,24 min per ep,9.3,676737.0,1.0,PG-13 - Teens 13 or older,Madhouse,"Aniplex, Dentsu, Shogakukan-Shueisha Productio...",https://cdn.myanimelist.net/images/anime/1015/...,https://www.youtube.com/watch?v=ZEkwCGJ3o7M,During their decade-long quest to defeat the D...
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy",TV,64.0,Finished Airing,2009-04-05T00:00:00+00:00,2010-07-04T00:00:00+00:00,24 min per ep,9.1,2223666.0,2.0,R - 17+ (violence & profanity),Bones,"Aniplex, Square Enix, Mainichi Broadcasting Sy...",https://cdn.myanimelist.net/images/anime/1208/...,https://www.youtube.com/watch?v=1ac3_YdSSy0,After a horrific alchemy experiment goes wrong...


In [5]:
anime.drop(columns = ['status',"aired_from","aired_to","scored_by","rank","score","image","trailer","producers","duration_per_ep","episodes"],inplace = True)

In [6]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28825 entries, 0 to 28824
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        28825 non-null  int64 
 1   name      28825 non-null  object
 2   genres    22756 non-null  object
 3   type      28738 non-null  object
 4   rating    28101 non-null  object
 5   studios   16936 non-null  object
 6   synopsis  23457 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.5+ MB


In [7]:
anime.isnull().sum()

id              0
name            0
genres       6069
type           87
rating        724
studios     11889
synopsis     5368
dtype: int64

In [8]:
# removed null values
anime.dropna(inplace=True)

In [9]:
anime.duplicated().sum()

np.int64(77)

In [10]:
# reomove duplicate values
anime.drop_duplicates(inplace=True)

In [11]:
anime['genres'] = anime['genres'].astype(str).str.strip('[]').str.replace(',,', ',').str.split(',')

In [12]:
anime["genres"]

0                 [Adventure,  Drama,  Fantasy]
1        [Action,  Adventure,  Drama,  Fantasy]
2                   [Drama,  Sci-Fi,  Suspense]
3                   [Action,  Drama,  Suspense]
4                    [Action,  Comedy,  Sci-Fi]
                          ...                  
28797              [Comedy,  Fantasy,  Mystery]
28798                        [Comedy,  Fantasy]
28801                                  [Comedy]
28806                        [Comedy,  Fantasy]
28821     [Drama,  Fantasy,  Mystery,  Romance]
Name: genres, Length: 14058, dtype: object

In [13]:
# string to list conversion
anime['type'] = anime['type'].apply(lambda x:x.split())
anime['rating'] = anime['rating'].apply(lambda x:x.split())
anime['studios'] = anime['studios'].apply(lambda x:x.split())
anime['synopsis'] = anime['synopsis'].apply(lambda x:x.split())

In [14]:
anime.head()

,id,name,genres,type,rating,studios,synopsis
0,52991,Frieren: Beyond Journey's End,"[Adventure, Drama, Fantasy]",[TV],"[PG-13, -, Teens, 13, or, older]",[Madhouse],"[During, their, decade-long, quest, to, defeat..."
1,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy]",[TV],"[R, -, 17+, (violence, &, profanity)]",[Bones],"[After, a, horrific, alchemy, experiment, goes..."
2,9253,Steins;Gate,"[Drama, Sci-Fi, Suspense]",[TV],"[PG-13, -, Teens, 13, or, older]","[White, Fox]","[Eccentric, scientist, Rintarou, Okabe, has, a..."
3,38524,Attack on Titan Season 3 Part 2,"[Action, Drama, Suspense]",[TV],"[R, -, 17+, (violence, &, profanity)]","[Wit, Studio]","[Seeking, to, restore, humanity's, diminishing..."
4,28977,Gintama Season 4,"[Action, Comedy, Sci-Fi]",[TV],"[PG-13, -, Teens, 13, or, older]","[Bandai, Namco, Pictures]","[Gintoki,, Shinpachi,, and, Kagura, return, as..."


In [15]:
anime['genres'] = anime['genres'].apply(lambda x:[i.replace(" ","")for i in x])
anime['type'] = anime['type'].apply(lambda x:[i.replace(" ","")for i in x])
anime['rating'] = anime['rating'].apply(lambda x:[i.replace(" ","")for i in x])
anime['studios'] = anime['studios'].apply(lambda x:[i.replace(" ","")for i in x])

In [16]:
anime.head()

,id,name,genres,type,rating,studios,synopsis
0,52991,Frieren: Beyond Journey's End,"[Adventure, Drama, Fantasy]",[TV],"[PG-13, -, Teens, 13, or, older]",[Madhouse],"[During, their, decade-long, quest, to, defeat..."
1,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy]",[TV],"[R, -, 17+, (violence, &, profanity)]",[Bones],"[After, a, horrific, alchemy, experiment, goes..."
2,9253,Steins;Gate,"[Drama, Sci-Fi, Suspense]",[TV],"[PG-13, -, Teens, 13, or, older]","[White, Fox]","[Eccentric, scientist, Rintarou, Okabe, has, a..."
3,38524,Attack on Titan Season 3 Part 2,"[Action, Drama, Suspense]",[TV],"[R, -, 17+, (violence, &, profanity)]","[Wit, Studio]","[Seeking, to, restore, humanity's, diminishing..."
4,28977,Gintama Season 4,"[Action, Comedy, Sci-Fi]",[TV],"[PG-13, -, Teens, 13, or, older]","[Bandai, Namco, Pictures]","[Gintoki,, Shinpachi,, and, Kagura, return, as..."


In [17]:
anime['tags'] = anime['genres'] + anime['type'] + anime['rating'] + anime['studios'] + anime['synopsis']

In [18]:
anime.head()

,id,name,genres,type,rating,studios,synopsis,tags
0,52991,Frieren: Beyond Journey's End,"[Adventure, Drama, Fantasy]",[TV],"[PG-13, -, Teens, 13, or, older]",[Madhouse],"[During, their, decade-long, quest, to, defeat...","[Adventure, Drama, Fantasy, TV, PG-13, -, Teen..."
1,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy]",[TV],"[R, -, 17+, (violence, &, profanity)]",[Bones],"[After, a, horrific, alchemy, experiment, goes...","[Action, Adventure, Drama, Fantasy, TV, R, -, ..."
2,9253,Steins;Gate,"[Drama, Sci-Fi, Suspense]",[TV],"[PG-13, -, Teens, 13, or, older]","[White, Fox]","[Eccentric, scientist, Rintarou, Okabe, has, a...","[Drama, Sci-Fi, Suspense, TV, PG-13, -, Teens,..."
3,38524,Attack on Titan Season 3 Part 2,"[Action, Drama, Suspense]",[TV],"[R, -, 17+, (violence, &, profanity)]","[Wit, Studio]","[Seeking, to, restore, humanity's, diminishing...","[Action, Drama, Suspense, TV, R, -, 17+, (viol..."
4,28977,Gintama Season 4,"[Action, Comedy, Sci-Fi]",[TV],"[PG-13, -, Teens, 13, or, older]","[Bandai, Namco, Pictures]","[Gintoki,, Shinpachi,, and, Kagura, return, as...","[Action, Comedy, Sci-Fi, TV, PG-13, -, Teens, ..."


In [19]:
new_df = anime[['id','name','tags']]

In [20]:
# list to string conversion
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [21]:
new_df['tags'][1]

'Action Adventure Drama Fantasy TV R - 17+ (violence & profanity) Bones After a horrific alchemy experiment goes wrong in the Elric household, brothers Edward and Alphonse are left in a catastrophic new reality. Ignoring the alchemical principle banning human transmutation, the boys attempted to bring their recently deceased mother back to life. Instead, they suffered brutal personal loss: Alphonse\'s body disintegrated while Edward lost a leg and then sacrificed an arm to keep Alphonse\'s soul in the physical realm by binding it to a hulking suit of armor. The brothers are rescued by their neighbor Pinako Rockbell and her granddaughter Winry. Known as a bio-mechanical engineering prodigy, Winry creates prosthetic limbs for Edward by utilizing "automail," a tough, versatile metal used in robots and combat armor. After years of training, the Elric brothers set off on a quest to restore their bodies by locating the Philosopher\'s Stone—a powerful gem that allows an alchemist to defy the 

In [22]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [23]:
new_df.head()

,id,name,tags
0,52991,Frieren: Beyond Journey's End,adventure drama fantasy tv pg-13 - teens 13 or...
1,5114,Fullmetal Alchemist: Brotherhood,action adventure drama fantasy tv r - 17+ (vio...
2,9253,Steins;Gate,drama sci-fi suspense tv pg-13 - teens 13 or o...
3,38524,Attack on Titan Season 3 Part 2,action drama suspense tv r - 17+ (violence & p...
4,28977,Gintama Season 4,action comedy sci-fi tv pg-13 - teens 13 or ol...


## Steming

In [24]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [25]:
def stem(text):
    y =[]

    for i in text.split():
        y.append(ps.stem(i))

    return" ".join(y) 

In [26]:
new_df['tags'] = new_df['tags'].apply(stem)

In [27]:
new_df['tags'][0]

'adventur drama fantasi tv pg-13 - teen 13 or older madhous dure their decade-long quest to defeat the demon king, the member of the hero\' party—himmel himself, the priest heiter, the dwarf warrior eisen, and the elven mage frieren—forg bond through adventur and battles, creat unforgett preciou memori for most of them. however, the time that frieren spend with her comrad is equival to mere a fraction of her life, which ha last over a thousand years. when the parti disband after their victory, frieren casual return to her "usual" routin of collect spell across the continent. due to her differ sens of time, she seemingli hold no strong feel toward the experi she went through. as the year pass, frieren gradual realiz how her day in the hero\' parti truli impact her. wit the death of two of her former companions, frieren begin to regret have taken their presenc for granted; she vow to better understand human and creat real person connections. although the stori of that onc memor journey h

In [28]:
new_df['tags'][1]

'action adventur drama fantasi tv r - 17+ (violenc & profanity) bone after a horrif alchemi experi goe wrong in the elric household, brother edward and alphons are left in a catastroph new reality. ignor the alchem principl ban human transmutation, the boy attempt to bring their recent deceas mother back to life. instead, they suffer brutal person loss: alphonse\' bodi disintegr while edward lost a leg and then sacrif an arm to keep alphonse\' soul in the physic realm by bind it to a hulk suit of armor. the brother are rescu by their neighbor pinako rockbel and her granddaught winry. known as a bio-mechan engin prodigy, winri creat prosthet limb for edward by util "automail," a tough, versatil metal use in robot and combat armor. after year of training, the elric brother set off on a quest to restor their bodi by locat the philosopher\' stone—a power gem that allow an alchemist to defi the tradit law of equival exchange. as edward becom an infam alchemist and gain the nicknam "fullmeta

## Vectorization

In [29]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000,stop_words='english')

In [30]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [31]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [32]:
cv.get_feature_names_out()

array(['000', '10', '100', ..., 'zoid', 'zombi', 'zone'],
      shape=(5000,), dtype=object)

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
# calculating cosine distance between vectors 
similarity = cosine_similarity(vectors)

In [35]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(2992, np.float64(0.3489772473629054)),
 (4913, np.float64(0.33978569616828225)),
 (13090, np.float64(0.33588255306372117)),
 (6200, np.float64(0.3069800647022711)),
 (5763, np.float64(0.30236773884142537))]

In [36]:
def recommend(anime):
    anime_index = new_df[new_df['name'] == anime].index[0]
    distances = similarity[anime_index]
    anime_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

    for i in anime_list:
         print(new_df.iloc[i[0]]['name'])
    return

In [37]:
recommend("Naruto")

My Hero Academia: Training of the Dead
My Hero Academia: Rescue! Rescue Training
My Hero Academia: Heroes Rising
My Hero Academia Season 4
How I Attended an All-Guy's Mixer


In [38]:
import pickle

In [39]:
pickle.dump(new_df.to_dict(),open('anime_dict.pkl','wb'))

In [40]:
new_df.to_dict()

{'id': {0: 52991,
  1: 5114,
  2: 9253,
  3: 38524,
  4: 28977,
  5: 60022,
  6: 39486,
  7: 11061,
  8: 9969,
  9: 15417,
  10: 820,
  11: 41467,
  12: 34096,
  13: 43608,
  14: 42938,
  15: 4181,
  16: 918,
  17: 28851,
  18: 2904,
  19: 35180,
  20: 15335,
  21: 19,
  22: 58514,
  23: 37491,
  24: 54492,
  25: 51535,
  26: 35247,
  27: 37987,
  28: 40682,
  29: 32281,
  30: 59571,
  32: 49387,
  33: 36838,
  34: 55255,
  35: 37510,
  36: 31758,
  37: 40028,
  38: 263,
  39: 32935,
  40: 37521,
  41: 199,
  42: 17074,
  43: 48583,
  44: 2921,
  45: 47917,
  46: 51009,
  47: 1,
  48: 55690,
  49: 52198,
  50: 50160,
  51: 21,
  52: 58567,
  55: 24701,
  56: 45649,
  57: 44074,
  58: 48569,
  59: 47778,
  60: 52215,
  61: 39894,
  62: 50172,
  63: 53998,
  64: 33095,
  65: 1575,
  66: 44,
  67: 21939,
  68: 33352,
  69: 245,
  70: 40434,
  71: 57864,
  72: 56784,
  73: 5258,
  74: 431,
  75: 164,
  76: 33050,
  77: 49413,
  78: 457,
  79: 50399,
  80: 58125,
  81: 2001,
  82: 46102,
  

In [41]:
pickle.dump(similarity,open('similarity.pkl','wb'))

In [ ]:
import pickle
from sklearn.metrics.pairwise import cosine_similarity

recommendations = {}

for i in range(len(new_df)):

    distances = cosine_similarity(
        vectors[i].reshape(1,-1),
        vectors
    ).flatten()

    anime_list = sorted(
        list(enumerate(distances)),
        key=lambda x:x[1],
        reverse=True
    )[1:6]

    recommendations[new_df.iloc[i]['name']] = [
        new_df.iloc[j[0]]['name']
        for j in anime_list
    ]

pickle.dump(
    recommendations,
    open("recommendations.pkl","wb")
)

print("Done")